# 主要代号表

| 代号（机械设计手册） | 代号（市川常雄） | 意义 | 单位 |
| :----: | :----: | :----: | :----: |
| $a$ | $A$ | 中心距，标准齿轮及高度变位齿轮的中心距 | mm |
| $b$ | $b$ | 齿宽 | mm |
| $i$ | $i$ | 传动比 | |
| $p_b$ | $t_0$ | 传动比 | |
| $r_w$ ($r^\prime$) | $R$ | 节圆半径 | mm |
| $r_a$ | $R_2$ | 齿顶圆半径 | mm |
| $r_{a1}$, $r_{a2}$ | $R_2$, $R_2^\prime$ | 主动轮、被动轮齿顶圆半径 | mm |
| $r_b$ | $R_g$ | 基圆半径 | mm |
| $z$ | $z$ | 齿数 | |
| $z_1$, $z_2$ | $z$, $z^\prime$ | 主动轮、被动轮齿数 | |
| $a_w$ ($a^\prime$) | | 角度变位齿轮的中心距 | mm |
| $\alpha$ | | 压力角，齿廓角 | ($^\circ$), rad |
| $\alpha_n$  | $\alpha_n$  | 法向分度圆压力角 | ($^\circ$), rad |
| $\alpha^\prime$ | $\alpha$ | 啮合角 | ($^\circ$), rad |
| $\alpha_a$ | | 齿顶压力角 | ($^\circ$), rad |
| $\alpha_{a1}$, $\alpha_{a2}$ | | 主动轮、被动轮齿顶压力角 | ($^\circ$), rad |
| $\beta$ | $\beta$ | 分度圆螺旋角，端面齿廓角 | ($^\circ$), rad |
| $\beta_b$ | $\beta_g$ | 基圆螺旋角 | ($^\circ$), rad |
| $\epsilon$ | | 重合度 | |

成大先. 机械设计手册[M]. 第6版. 第3卷. 北京: 化学工业出版社, 2016: 15-3--15-8.

$$
\epsilon = \frac{1}{2 \pi} \left[
    z_1 \left( \tan \alpha_{a1} - \tan \alpha' \right) 
\pm z_2 \left( \tan \alpha_{a2} - \tan \alpha' \right)
\right] \tag {1}
$$
成大先. 机械设计手册[M]. 第6版. 第3卷. 北京: 化学工业出版社, 2016: 15-51.

$$
\cos \alpha_a = \frac{r_b}{r_a}  \tag{2}
$$

$$
a' \cos \alpha' = a \cos \alpha \tag {3}
$$

$$
V_{th} = \pi b \left \{
    {R_2}^2 - R^2 
    \pm i ({{R_2}^\prime}^2 - {R^\prime}^2) \\
    - \frac{1}{12} (1 \pm i) {t_0}^2
    - \frac{b^2}{12} (1 \pm i) \tan^2 \beta_g
\right \}
$$

In [24]:
from sympy import symbols, cos, tan, acos, rad, pi
from sympy.abc import alpha

z = symbols('z_:2', positive=True, integer=True)
m = symbols('m', positive=True, rational=True)
b = symbols('b', positive=True, rational=True)

r_a = [(z_i + 2) * m / 2 for z_i in z]
r = [z_i * m / 2 for z_i in z]
r_b = [r_i * cos(alpha) for r_i in r]
p_b = pi * m * cos(alpha)

a = sum( r_i for r_i in r)
a_prime = symbols('a^prime')

alpha_a = [acos(r_bi / r_ai) for r_bi,  r_ai in zip(r_b, r_a)] #2
alpha_prime = acos(a * cos(alpha) / a_prime) #(3)

epsilon = sum( #(1)
    (z_i * (tan(phi_i) - tan(alpha_prime))) / (2 * pi)
    for z_i, phi_i in zip(z, alpha_a)
).simplify()

epsilon.free_symbols

{a^prime, alpha, m, z_0, z_1}

In [25]:
epsilon.subs(
    {
        z[0]: 9,
        z[1]: 9,
        m: 1.75,
        a_prime: 15.8,
        alpha: rad(20), #分度圆压力角
    }
).evalf()

1.31197310662698

重合度小于等于1.5，方可设计无侧隙单卸荷槽

In [26]:
k_c = 3 * epsilon**2 - 6 * epsilon + 4

q = sum(pi * b * (r_ai**2 - ri**2 - k_c * p_b**2 / 12) for r_ai, ri in zip(r_a, r))

q.free_symbols

{a^prime, alpha, b, m, z_0, z_1}

In [27]:
q_mm3 = q.subs(
    {
        z[0]: 9,
        z[1]: 9,
        m: 1.75,
        b: 15,
        a_prime: 15.8,
        alpha: rad(20), #分度圆压力角
    }
).evalf()

In [34]:
n_max = (60 * 1000 * 4 / pi / r_a[0])
(n_max * q).subs(
    {
        z[0]: 9,
        z[1]: 9,
        m: 1.75,
        b: 15,
        a_prime: 15.8,
        alpha: rad(20), #分度圆压力角
    }
).evalf() / 1000000

20.7595073296371